In [1]:
## IMPORT

import warnings
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf           # for OLS (smf.ols)
from sklearn.metrics import mean_squared_error  # perform mean-squared for RMSE
from statsmodels.stats.anova import anova_lm    # perform Chi-Squared test

## IGNORE WARNINGS

warnings.filterwarnings("ignore")

## LOAD DATA

train = pd.read_csv("data/GDP_Forecast_train.csv")
test = pd.read_csv("data/GDP_PCA_for_test.csv")
# test = pd.read_csv("data/GDP_Forecast_test.csv")

## DEFINE FUNCTIONS FOR REUSE

def calc_performance(formula1, model1, formula2, model2):
    print(pd.DataFrame({'model'   : [formula1, formula2],
                    'Adj.R^2' : [model1.rsquared_adj, model2.rsquared_adj]}))
    print()
    print(anova_lm(model1, model2, test = "Chisq"))

def create_submission(test_df, y_pred, filename):
    df = pd.DataFrame({"YQ": test_df["YQ"], "NGDP": y_pred})
    df.to_csv(f"attempts/{filename}.csv", index = False)

## CONVERT YEAR-QUARTER: SPLIT COLUMNS
# we are using a different dataset for test because we want to calculate RMSE before submitting

# ensure year is an integer
train["year"] = train["YQ"].str[:4].astype(int)
# keep qtr as string / category
train["qtr"] = train["YQ"].str[4:]

## USING SEPARATE TEST DATASET FOR TESTING RMSE

# convert the data to YQ, year and quarter
test["observation_date"] = pd.to_datetime(test["observation_date"], dayfirst = True)
test["year"] = test["observation_date"].dt.year
test["qtr"] = "Q" + test["observation_date"].dt.quarter.astype(str)
test["YQ"] = test["year"].astype(str) + test["qtr"]
# remove observation_date column and move GDP to the back while changing col name to NGDP
test.drop(columns = "observation_date", inplace = True)
col = test.pop("GDP_PCA")
test["NGDP"] = col
# get the quarterly
test.sort_values(by = "YQ")
# # check data for 1990 - 2015 is similar to train to continue with this dataset for test
# test[test["year"] <= 2015].transpose()
## NOTE: data from later years is not as accurate, but we will use this to tentatively test our RMSE
test = test[test["year"]>= 2016].reset_index(drop = True)
test.head()

,year,qtr,YQ,NGDP
0,2016,Q1,2016Q1,2.0
1,2016,Q2,2016Q2,4.1
2,2016,Q3,2016Q3,3.9
3,2016,Q4,2016Q4,4.2
4,2017,Q1,2017Q1,4.1


In [2]:
mod1 = smf.ols("NGDP ~ year + qtr", train).fit()
print(mod1.summary())
# getting prediction values based off model
y_pred, y_true = mod1.predict(test), test["NGDP"]
print(f"\nRMSE: {np.sqrt(mean_squared_error(y_true, y_pred))}")
create_submission(test, y_pred, "0_baseline_mod1") # public score 11.58888

                            OLS Regression Results                            
Dep. Variable:                   NGDP   R-squared:                       0.118
Model:                            OLS   Adj. R-squared:                  0.083
Method:                 Least Squares   F-statistic:                     3.324
Date:                Fri, 06 Mar 2026   Prob (F-statistic):             0.0134
Time:                        14:27:44   Log-Likelihood:                -244.91
No. Observations:                 104   AIC:                             499.8
Df Residuals:                      99   BIC:                             513.0
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept    219.8404     68.424      3.213      0.0

It is recognised that multicollinearity could exist due to the condition number being large, this could be due to the QUARTER variable, seen in how the p-value for Q3 and Q4 are statistically not significant to NGDP. To prove this, the baseline model will be run without to see the difference in performance.

In [3]:
mod2 = smf.ols("NGDP ~ year", train).fit()
print(mod2.summary())
# getting prediction values based off model
y_pred, y_true = mod2.predict(test), test["NGDP"]
print(f"\nRMSE: {np.sqrt(mean_squared_error(y_true, y_pred))}")
# create_submission(test, y_pred, "0_baseline_mod2") # public score 11.58888
calc_performance("NGDP ~ year + qtr", mod1, "NGDP ~ year", mod2)

                            OLS Regression Results                            
Dep. Variable:                   NGDP   R-squared:                       0.089
Model:                            OLS   Adj. R-squared:                  0.080
Method:                 Least Squares   F-statistic:                     9.916
Date:                Fri, 06 Mar 2026   Prob (F-statistic):            0.00215
Time:                        14:27:44   Log-Likelihood:                -246.64
No. Observations:                 104   AIC:                             497.3
Df Residuals:                     102   BIC:                             502.6
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept    220.4192     68.539      3.216      0.0

Insights: It is recognised that the adjusted R-squared value moved insignificantly and the performance drop was small. RMSE dropped from 11.309452691382095 to 11.214203690382107, all pointing to `qtr` being an insignificant in predicting `NGDP`. This will be dropped to prevent overfitting and multicollinearity or other numerical problems.

### How Accounting Data affects GDP (COMPUSTAT on WRDS)
[Compustat North America](https://wrds-www.wharton.upenn.edu/pages/get-data/compustat-capital-iq-standard-poors/compustat/north-america-daily/) is a database of U.S. and Canadian fundamental and market information on active and inactive publicly held companies. It provides more than 300 annual and 100 quarterly Income Statement, Balance Sheet, Statement of Cash Flows, and supplemental data items. For most companies, annual history is available back to 1950 and quarterly history back to 1962 with monthly market history back to 1962.

- (conm) Company Name
- (revtq) The "Top Line" Signal: Revenue - Total
- (ppentq) The "Investment" Signal: Property Plant and Equipment - Total (Net)
- (invtq) The "Inventory" Signal: Inventories - Total
- (oibdpq) The "Operations" Signal: Operating Income Before Depreciation - Quarterly
- (xrdq) The "Innovation" Signal: Research and Development Expense

In [4]:
## LOAD DATA

compustat = pd.read_csv("data/compustat_quarterly_financials2.csv")
# dropping columns with no unique data + gvkey
compustat.drop(columns = ["costat", "curcdq", "datafmt", "indfmt", "consol", "gvkey", "datadate"], inplace = True)

## CREATING AGGREGATE TO GET QUARTERLY DATA

agg_dict = {"revtq": "sum", "invtq": "sum", "oibdpq": "sum", "ppentq": "sum", "xrdq": "sum"}
compustat_macro = compustat.groupby("datacqtr").agg(agg_dict).reset_index()
compustat_macro.sort_values(by = "datacqtr", ascending = True, inplace = True) # for shift

## CREATING GROWTH AND GROWTH LAG

for col in agg_dict.keys():
    # calculating respective lags
    compustat_macro[f"{col}_lag"] = compustat_macro[col].shift(1)
    # calculating respective growth rates
    compustat_macro[f"{col}_growth"] = compustat_macro[col].pct_change()
    # creating lags for forecasting
    compustat_macro[f"{col}_growth_lag"] = compustat_macro[f"{col}_growth"].shift(1)
# convert infinite values to NaN for .drop()
compustat_macro.replace([np.inf, -np.inf], np.nan, inplace = True)
# drop missing values
compustat_macro.dropna(inplace = True)

## MERGING COMPUSTAT DATA TO TRAIN AND TEST

train_comp_macro = train.merge(compustat_macro, how = "inner", left_on = "YQ", right_on = "datacqtr")
test_comp_macro = test.merge(compustat_macro, how = "inner", left_on = "YQ", right_on = "datacqtr")

## CREATING FORMULAS FOR MODELS

formula1 = "NGDP ~ year + revtq + invtq + oibdpq + ppentq + xrdq"
formula2 = "NGDP ~ year + revtq_lag + invtq_lag + oibdpq_lag + ppentq_lag + xrdq_lag"
formula3 = "NGDP ~ year + revtq_growth + invtq_growth + oibdpq_growth + ppentq_growth + xrdq_growth"
formula4 = "NGDP ~ year + revtq_growth_lag + invtq_growth_lag + oibdpq_growth_lag + ppentq_growth_lag + xrdq_growth_lag"

In [5]:
## RUNNING MODEL 1 ON PREDICTORS

comp_model1 = smf.ols(formula1, train_comp_macro).fit()
y_pred, y_true = comp_model1.predict(test_comp_macro), test_comp_macro["NGDP"]
print(f"\nRMSE on Predictors: {np.sqrt(mean_squared_error(y_true, y_pred))}")

## RUNNING MODEL 2 ON PREDICTOR LAGS

comp_model2 = smf.ols(formula2, train_comp_macro).fit()
y_pred, y_true = comp_model2.predict(test_comp_macro), test_comp_macro["NGDP"]
print(f"\nRMSE on Predictor Lags: {np.sqrt(mean_squared_error(y_true, y_pred))}")
calc_performance(formula1, comp_model1, formula2, comp_model2)

## RUNNING MODEL 3 ON PREDICTOR GROWTHS

comp_model3 = smf.ols(formula3, train_comp_macro).fit()
y_pred, y_true = comp_model3.predict(test_comp_macro), test_comp_macro["NGDP"]
print(f"\nRMSE on Predictor Growths: {np.sqrt(mean_squared_error(y_true, y_pred))}")
calc_performance(formula2, comp_model2, formula3, comp_model3)

## RUNNING MODEL 4 ON PREDICTOR GROWTH LAGS

comp_model4 = smf.ols(formula4, train_comp_macro).fit()
y_pred, y_true = comp_model4.predict(test_comp_macro), test_comp_macro["NGDP"]
print(f"\nRMSE on Predictor Growth Lags: {np.sqrt(mean_squared_error(y_true, y_pred))}")
calc_performance(formula3, comp_model3, formula4, comp_model4)

## Insights: Model Performance (best to worst)
## predictors (1) > predictor lags (2) > predictor growth lags (4) > predictor growths (3)


RMSE on Predictors: 10.7262427622723

RMSE on Predictor Lags: 11.099918823245151
                                               model   Adj.R^2
0  NGDP ~ year + revtq + invtq + oibdpq + ppentq ...  0.346900
1  NGDP ~ year + revtq_lag + invtq_lag + oibdpq_l...  0.218636

   df_resid         ssr  df_diff    ss_diff  Chisq  Pr(>Chisq)
0      96.0  459.387534      0.0        NaN    0.0         0.0
1      96.0  549.607940     -0.0 -90.220406    0.0         0.0

RMSE on Predictor Growths: 11.092666572821239
                                               model   Adj.R^2
0  NGDP ~ year + revtq_lag + invtq_lag + oibdpq_l...  0.218636
1  NGDP ~ year + revtq_growth + invtq_growth + oi...  0.134744

   df_resid        ssr  df_diff    ss_diff  Chisq  Pr(>Chisq)
0      96.0  549.60794      0.0        NaN    0.0         0.0
1      96.0  608.61690     -0.0 -59.008959    0.0         0.0

RMSE on Predictor Growth Lags: 10.83308909755563
                                               model   Adj.R^2
0  

In [6]:
print(comp_model1.summary())

                            OLS Regression Results                            
Dep. Variable:                   NGDP   R-squared:                       0.385
Model:                            OLS   Adj. R-squared:                  0.347
Method:                 Least Squares   F-statistic:                     10.03
Date:                Fri, 06 Mar 2026   Prob (F-statistic):           1.39e-08
Time:                        14:27:45   Log-Likelihood:                -223.15
No. Observations:                 103   AIC:                             460.3
Df Residuals:                      96   BIC:                             478.7
Df Model:                           6                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept   -565.9451    342.190     -1.654      0.1

In [7]:
print(comp_model2.summary())

                            OLS Regression Results                            
Dep. Variable:                   NGDP   R-squared:                       0.265
Model:                            OLS   Adj. R-squared:                  0.219
Method:                 Least Squares   F-statistic:                     5.757
Date:                Fri, 06 Mar 2026   Prob (F-statistic):           3.76e-05
Time:                        14:27:45   Log-Likelihood:                -232.39
No. Observations:                 103   AIC:                             478.8
Df Residuals:                      96   BIC:                             497.2
Df Model:                           6                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept  -1194.5296    399.182     -2.992      0.0

In [8]:
print(comp_model3.summary())

                            OLS Regression Results                            
Dep. Variable:                   NGDP   R-squared:                       0.186
Model:                            OLS   Adj. R-squared:                  0.135
Method:                 Least Squares   F-statistic:                     3.647
Date:                Fri, 06 Mar 2026   Prob (F-statistic):            0.00264
Time:                        14:27:45   Log-Likelihood:                -237.64
No. Observations:                 103   AIC:                             489.3
Df Residuals:                      96   BIC:                             507.7
Df Model:                           6                                         
Covariance Type:            nonrobust                                         
                    coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------
Intercept       197.1212     70.725      2.787

In [9]:
print(comp_model4.summary())

                            OLS Regression Results                            
Dep. Variable:                   NGDP   R-squared:                       0.226
Model:                            OLS   Adj. R-squared:                  0.177
Method:                 Least Squares   F-statistic:                     4.668
Date:                Fri, 06 Mar 2026   Prob (F-statistic):           0.000331
Time:                        14:27:45   Log-Likelihood:                -235.03
No. Observations:                 103   AIC:                             484.1
Df Residuals:                      96   BIC:                             502.5
Df Model:                           6                                         
Covariance Type:            nonrobust                                         
                        coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------
Intercept           174.0474     69.46

It is recognised in the models that Revenue seems to not be statistically significant to the Nominal GDP Growth Rate, which is against our mental model as we believe that Revenue should translate to GDP Growth Rate. However, it is recognised that it is mainly Tax Revenue that affects GDP.

In [10]:
## FINETUNE MODEL - TAKING LOWEST P-VALUES:

formula5 = "NGDP ~ year + revtq_lag + xrdq_growth_lag + invtq + ppentq + oibdpq"
formula6 = "NGDP ~ year + revtq_lag + xrdq_growth_lag + invtq + ppentq + oibdpq_lag"

## RUNNING MODEL 5 ON PREDICTORS + OPS INC

comp_model5 = smf.ols(formula5, train_comp_macro).fit()
y_pred, y_true = comp_model5.predict(test_comp_macro), test_comp_macro["NGDP"]
print(f"\nRMSE on Predictors + Ops Inc: {np.sqrt(mean_squared_error(y_true, y_pred))}")
# proving model5 better than model1 (prev best)
calc_performance(formula1, comp_model1, formula5, comp_model5)

## RUNNING MODEL 6 ON PREDICTORS + OPS INC LAG

comp_model6 = smf.ols(formula6, train_comp_macro).fit()
y_pred, y_true = comp_model6.predict(test_comp_macro), test_comp_macro["NGDP"]
print(f"\nRMSE on Predictors + Ops Inc Lag: {np.sqrt(mean_squared_error(y_true, y_pred))}")
# checking if ops income is better or ops income lag (because both p-value <0.001)
calc_performance(formula5, comp_model5, formula6, comp_model6)

## CREATING CSV OUTPUT TO RUN ON KAGGLE TO GET PUBLIC SCORE

create_submission(test_comp_macro, y_pred, "1_compustat_model5") # public score 10.94454
create_submission(test_comp_macro, y_pred, "2_compustat_model6") # public score 10.94454


RMSE on Predictors + Ops Inc: 10.691896851663365
                                               model   Adj.R^2
0  NGDP ~ year + revtq + invtq + oibdpq + ppentq ...  0.346900
1  NGDP ~ year + revtq_lag + xrdq_growth_lag + in...  0.334029

   df_resid         ssr  df_diff   ss_diff  Chisq  Pr(>Chisq)
0      96.0  459.387534      0.0       NaN    0.0         0.0
1      96.0  468.440832     -0.0 -9.053298    0.0         0.0

RMSE on Predictors + Ops Inc Lag: 10.61893708752085
                                               model   Adj.R^2
0  NGDP ~ year + revtq_lag + xrdq_growth_lag + in...  0.334029
1  NGDP ~ year + revtq_lag + xrdq_growth_lag + in...  0.129873

   df_resid         ssr  df_diff     ss_diff  Chisq  Pr(>Chisq)
0      96.0  468.440832      0.0         NaN    0.0         0.0
1      96.0  612.043279     -0.0 -143.602447    0.0         0.0


It is recognised that the RMSE on the updated model has improved from 10.7262427622723 (`comp_model1`) to 10.691896851663365 (`comp_model5`) and 10.61893708752085 (`comp_model6`).

While the private RMSE favours Ops Inc Lag (`comp_model6`), the difference is marginal, and both models scored identally on Kaggle (10.94454). The higher Adjusted R-Squared of `comp_model5` (0.334029) compared to `comp_model6` (0.129873) and its lower training SSR reflects better in-sample fit. But this is likely because choosing current quarter's Ops Inc is contemporaneous with GDP, making the in-sample fit artificially strong rather than reflecting actual predictive power.

It is recognised that manually mixing feature representation based on p-values is not a principled approach to model selection. As such we will proceed to conduct regularisation (*Ridge, Lasso, ElasticNet*) to prevent overfitting, manage multicollinearity and improve out-of-sample generalisation, all of which have been issues in using OLS on this dataset.

The Compustat data is generally cleaned before publishing, so the previous quarter data should be considered rather than current data.

In [36]:
## IMPORT

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LassoCV, RidgeCV, ElasticNetCV

## STORING X AND Y FOR TRAIN AND TEST

features = [x for x in train_comp_macro.columns.tolist() if "lag" in x]

X_train = train_comp_macro[features]
y_train = train_comp_macro["NGDP"]
X_test = test_comp_macro[features]
y_test = test_comp_macro["NGDP"]

## SCALING BEFORE REGULARISATION (FEATURES ARE ON DIFFERENT SCALES)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)        # fit on train only, transform test

## FITTING LASSO MODEL ON TRAINING SET

lasso = LassoCV(cv = 5)
lasso.fit(X_train_scaled, y_train)
rmse_lasso = np.sqrt(mean_squared_error(y_test, lasso.predict(X_test_scaled)))
print(f"Lasso   | RMSE: {rmse_lasso:.4f}    | alpha: {lasso.alpha_}")

## FITTING RIDGE MODEL ON TRAINING SET

ridge = RidgeCV(cv = 5)
ridge.fit(X_train_scaled, y_train)
rmse_ridge = np.sqrt(mean_squared_error(y_test, ridge.predict(X_test_scaled)))
print(f"Ridge   | RMSE: {rmse_ridge:.4f}    | alpha: {ridge.alpha_}")

## FITTING ELASTICNET MODEL ON TRAINING SET

enet = ElasticNetCV(l1_ratio = [0.1, 0.5, 0.9], cv = 5)
enet.fit(X_train_scaled, y_train)
rmse_enet = np.sqrt(mean_squared_error(y_test, enet.predict(X_test_scaled)))
print(f"ElasNet | RMSE: {rmse_enet:.4f}    | alpha: {enet.alpha_}   | l1_ratio: {enet.l1_ratio_}")

## COMPARING FEATURE WEIGHTS ACROSS MODELS

print("\nFeature coefficients:")
print(f"{'Feature':<25} {'Lasso':>10} {'Ridge':>10} {'ElasNet':>10}")
for feat, lc, rc, ec in zip(features, lasso.coef_, ridge.coef_, enet.coef_):
    print(f"{feat:<25} {lc:>10.4f} {rc:>10.4f} {ec:>10.4f}")

Lasso   | RMSE: 11.1985    | alpha: 0.25593564914553335
Ridge   | RMSE: 11.2110    | alpha: 10.0
ElasNet | RMSE: 11.1728    | alpha: 0.9635812897026922   | l1_ratio: 0.1

Feature coefficients:
Feature                        Lasso      Ridge    ElasNet
revtq_lag                    -0.0000    -0.1967    -0.1144
revtq_growth_lag              0.0000     0.0346     0.0000
invtq_lag                    -0.0000    -0.3487    -0.1025
invtq_growth_lag              0.0000     0.3519     0.0059
oibdpq_lag                   -0.0000     0.8499    -0.0000
oibdpq_growth_lag             0.0000     0.0977     0.0000
ppentq_lag                   -0.3202    -0.7450    -0.1948
ppentq_growth_lag             0.0000    -0.0603     0.0000
xrdq_lag                     -0.3182    -0.4097    -0.2157
xrdq_growth_lag               0.0000    -0.3980    -0.0000


Insights:
- ppentq_lag and xrdq_lag exhibit strong consistent signals: all negative, non-zero
    - (ppentq_lag): Property, Plant and Equipment of last quarter predicts lower Nominal GDP Growth
    - (xrdq_lag): Research and Development spendings last quarter predicts lower Nominal GDP Growth
- revtq_lag and invtq_lag exhibit moderate signals: all negative
- oibdpq_lag exhibit inconsistent signals: The disagreement between Lasso and ElasNet (zero) and Ridge (large positive) suggests multicollinearity with other features. This is because Ridge regression retains all predictors, shrinking coefficients while Lasso picks one and zeros the rest.
- All growth lags seem to be mostly noise: Lasso mostly zeros out, Ridge and ElasNet are near-zero

Overall Insights: With just accounting data alone, the model performs poorly (RMSE ~11.2), suggesting accounting data is insufficient as a standalone predictor of Nominal GDP Growh. As such, the next step would be to add macro data from CRSP.

Types of models:
- OLS (smf.ols)
- Lasso
- Ridge
- Elastic Net

Things to consider:
- p-value: <0.05 (and coefficient fits mental model)
- r-squared: explained variation / total variation (high R^2 = model fits well)
- adjusted r-squared: downweights R^2 to make it unbiased (variation)
- test statistics
    - test coeff: t-test (more common, unknown pop_n s.d.) or z-test (known pop_n s.d)
    - test model as a whole: f-test (check adj. R^2 as well)
    - testing across models: chi-squared (chisq)
    - testing across models under ML concept: prediction errors - RMSE and MAE

